# Módulo 01 · Aula 3: Busca vetorial

**Duração:** 45 min

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- explicar o que é um *vector store* (documentos + vetores + busca pelos vizinhos mais próximos);
- usar o `InMemoryVectorStore` do langchain-core com `similarity_search` e `similarity_search_with_score`;
- criar um índice **FAISS** reaproveitando vetores já calculados, e explicar por que o projeto o importa de
  `src.rag.vectorstore`;
- interpretar a **distância L2** do FAISS e convertê-la em similaridade de cosseno;
- escolher `k` e **filtrar por metadados** (ex.: só políticas em vigor);
- salvar e carregar um índice com `save_local`/`load_local`, entendendo o alerta de `allow_dangerous_deserialization`;
- usar `build_index`, `save_index`, `load_index` e `index_info` de [src/rag/vectorstore.py](../src/rag/vectorstore.py).

## Pré-requisitos

- Aula 01.2 (embeddings, `embed_query`, similaridade de cosseno, regra de ouro do modelo).

## O que vamos construir

Na aula anterior, comparamos a pergunta com cada frase em um laço. Um **índice vetorial** faz isso por nós, com
qualquer número de trechos, e ainda guarda os metadados de cada um:

```mermaid
flowchart LR
    subgraph IDX[Índice vetorial]
        direction TB
        T1["trecho 1 · vetor · metadados"]
        T2["trecho 2 · vetor · metadados"]
        T3["..."]
    end
    Q([pergunta]) --> E[embed_query] --> S{k vizinhos<br/>mais próximos<br/>+ filtro}
    IDX --> S
    S --> R([top-k trechos<br/>com score])
```

> 💰 **Economia:** calculamos os vetores dos trechos **uma vez** (seção 1) e os reaproveitamos no FAISS. As perguntas
> também viram vetor uma única vez e são reutilizadas com os métodos `..._by_vector`.

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

## 1. `InMemoryVectorStore`: o índice mais simples

Um *vector store* guarda, para cada trecho, três coisas: o **texto**, o **vetor** e os **metadados**. E responde a uma
pergunta: "quais são os `k` trechos mais próximos deste vetor?". O langchain-core traz uma implementação em memória,
sem dependências, ótima para aprender e para testes.

Vamos usar doze trechos curtos, tirados das políticas da TechNova. Um deles vem da política de frete **arquivada**,
de 2024. Os metadados dizem de onde veio cada trecho e se ele está em vigor.

In [2]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from src.models import get_embeddings

embeddings = get_embeddings()

SNIPPETS = [
    ("policies/exchange.md", "active",
     "Trocas podem ser solicitadas em até 30 dias corridos após a entrega, com o produto sem sinais de uso."),
    ("policies/exchange.md", "active", "O frete da troca é gratuito, tanto na ida quanto no reenvio do novo produto."),
    ("policies/refund.md", "active",
     "Após a aprovação de um atendente humano, o valor é estornado em até 10 dias úteis, na mesma forma de pagamento."),
    ("policies/refund.md", "active",
     "Por arrependimento, o reembolso pode ser pedido em até 30 dias após a entrega, com o produto completo."),
    ("policies/shipping.md", "active", "Frete grátis para compras acima de R$ 299,00 na modalidade Normal."),
    ("policies/shipping.md", "active",
     "A entrega Expressa leva de 1 a 2 dias úteis nas capitais do Sudeste e do Sul e é sempre paga."),
    ("policies/shipping_2024.md", "archived", "Frete grátis para compras acima de R$ 199,00, em todo o Brasil."),
    ("policies/warranty.md", "active",
     "Todos os produtos têm 12 meses de garantia contra defeitos de fabricação, contados a partir da entrega."),
    ("policies/warranty.md", "active",
     "Danos por queda e contato com líquidos não são cobertos, exceto em produtos com certificação de resistência."),
    ("policies/payment.md", "active", "O Pix dá 5% de desconto sobre o valor dos produtos e o QR Code vale 30 minutos."),
    ("policies/payment.md", "active", "Compras acima de R$ 500,00 podem ser parceladas em até 10x sem juros no cartão."),
    ("policies/cancellation.md", "active", "Pedidos ainda não enviados podem ser cancelados sem custo em Minha Conta."),
]
docs = [Document(page_content=text, metadata={"source": source, "status": status}) for source, status, text in SNIPPETS]

memory_store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)  # UMA chamada: embed_documents
print(len(memory_store.store), "trechos no índice")

first_id, first_entry = next(iter(memory_store.store.items()))
print("Cada entrada guarda:", list(first_entry), "| vetor com", len(first_entry["vector"]), "dimensões")

2026-09-27 21:04:53 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


12 trechos no índice
Cada entrada guarda: ['id', 'vector', 'text', 'metadata'] | vetor com 3072 dimensões


**Observe:** `from_documents` chamou `embed_documents` com os doze textos de uma vez e guardou tudo em um dicionário
(`memory_store.store`), com um id gerado para cada trecho. Agora, a busca:

In [3]:
question = "Posso devolver um produto que não gostei?"
for doc in memory_store.similarity_search(question, k=2):
    print(f"- [{doc.metadata['source']}] {doc.page_content}")

- [policies/refund.md] Por arrependimento, o reembolso pode ser pedido em até 30 dias após a entrega, com o produto completo.
- [policies/exchange.md] Trocas podem ser solicitadas em até 30 dias corridos após a entrega, com o produto sem sinais de uso.


`similarity_search` devolve só os documentos. Para saber **quão** próximos eles estão, use
`similarity_search_with_score`. No `InMemoryVectorStore`, o score é a **similaridade de cosseno** (maior é melhor),
a mesma da aula anterior.

In [4]:
for doc, score in memory_store.similarity_search_with_score(question, k=4):
    print(f"{score:.3f}  [{doc.metadata['source']}] {doc.page_content[:70]}...")

0.591  [policies/refund.md] Por arrependimento, o reembolso pode ser pedido em até 30 dias após a ...
0.511  [policies/exchange.md] Trocas podem ser solicitadas em até 30 dias corridos após a entrega, c...
0.451  [policies/warranty.md] Todos os produtos têm 12 meses de garantia contra defeitos de fabricaç...
0.438  [policies/exchange.md] O frete da troca é gratuito, tanto na ida quanto no reenvio do novo pr...


Por dentro, não há mágica: é o mesmo cálculo que fizemos com numpy. Para provar, vamos refazer a busca à mão com os
vetores guardados no índice. Calculamos o vetor da pergunta **uma vez** e o reutilizamos daqui em diante.

In [5]:
import numpy as np

question_vector = embeddings.embed_query(question)  # guardado para reutilizar

entries = list(memory_store.store.values())
matrix = np.array([entry["vector"] for entry in entries])
scores = matrix @ np.array(question_vector)  # cosseno (vetores de norma 1)
top = np.argsort(-scores)[:4]

print("Busca à mão (numpy):")
for i in top:
    print(f"{scores[i]:.3f}  [{entries[i]['metadata']['source']}] {entries[i]['text'][:70]}...")

Busca à mão (numpy):
0.591  [policies/refund.md] Por arrependimento, o reembolso pode ser pedido em até 30 dias após a ...
0.511  [policies/exchange.md] Trocas podem ser solicitadas em até 30 dias corridos após a entrega, c...
0.451  [policies/warranty.md] Todos os produtos têm 12 meses de garantia contra defeitos de fabricaç...
0.438  [policies/exchange.md] O frete da troca é gratuito, tanto na ida quanto no reenvio do novo pr...


**Observe:** mesma ordem e mesmos scores. O `InMemoryVectorStore` compara a pergunta com **todos** os vetores, um a
um (busca exata por força bruta), em Python. Para dezenas ou centenas de trechos, é suficiente. Para um produto,
queremos algo mais rápido, que também saiba **salvar** o índice em disco com os metadados. É aí que entra o FAISS.

| | `InMemoryVectorStore` | FAISS |
|---|---|---|
| Onde roda | Python puro, em memória | biblioteca C++ otimizada, na própria máquina |
| Busca | força bruta | exata (`IndexFlatL2`) ou aproximada (IVF, HNSW) para milhões de vetores |
| Persistência | `dump`/`load` em JSON | `save_local`/`load_local` (arquivo binário + pickle) |
| Uso típico | aulas, testes, protótipos | produtos locais, como o nosso |

## 2. FAISS

**FAISS** (*Facebook AI Similarity Search*) é uma biblioteca de busca vetorial criada pela Meta. Ela roda na própria
máquina, sem servidor, e escala de centenas a bilhões de vetores. O LangChain oferece uma integração (a classe `FAISS`)
que junta o índice de vetores com os documentos e metadados.

### 2.1 Por que importamos de `src.rag.vectorstore`

A classe `FAISS` do LangChain vive no pacote **`langchain-community`**, que está sendo descontinuado: as integrações
estão migrando para pacotes dedicados (como o `langchain-openai`). Para não espalhar essa dependência pelo projeto,
**um único arquivo** importa do community: [src/rag/vectorstore.py](../src/rag/vectorstore.py). O resto do projeto,
inclusive os notebooks, importa `FAISS` de lá. Se o FAISS ganhar um pacote próprio, a mudança acontece em uma linha.

> ⚠️ Neste curso, nunca escreva `from langchain_community... import ...` em outro arquivo. Use sempre
> `from src.rag.vectorstore import FAISS`.

### 2.2 Criar o índice sem recalcular os vetores

Os vetores dos doze trechos já foram pagos. `FAISS.from_embeddings` recebe pares (texto, vetor) prontos e só usa o
cliente de embeddings mais tarde, para transformar as **perguntas** em vetor.

In [6]:
from src.rag.vectorstore import FAISS

faiss_index = FAISS.from_embeddings(
    text_embeddings=[(entry["text"], entry["vector"]) for entry in entries],  # vetores reaproveitados
    embedding=embeddings,  # usado só nas perguntas
    metadatas=[entry["metadata"] for entry in entries],
)
print("Tipo do índice FAISS:", type(faiss_index.index).__name__)
print("Vetores:", faiss_index.index.ntotal, "| dimensões:", faiss_index.index.d)

Tipo do índice FAISS: IndexFlatL2
Vetores: 12 | dimensões: 3072


**Observe:** `IndexFlatL2` é o índice **exato** do FAISS: compara com todos os vetores, como o `InMemoryVectorStore`,
mas em C++ e com instruções vetorizadas. O "L2" do nome é a métrica: **distância euclidiana**, e não cosseno.

## 3. Distância L2 x similaridade

Buscando a mesma pergunta no FAISS (com o vetor já calculado):

In [7]:
print("FAISS (distância L2 ao quadrado: MENOR é melhor):")
for doc, distance in faiss_index.similarity_search_with_score_by_vector(question_vector, k=4):
    print(f"{distance:.3f}  [{doc.metadata['source']}] {doc.page_content[:70]}...")

FAISS (distância L2 ao quadrado: MENOR é melhor):
0.818  [policies/refund.md] Por arrependimento, o reembolso pode ser pedido em até 30 dias após a ...
0.979  [policies/exchange.md] Trocas podem ser solicitadas em até 30 dias corridos após a entrega, c...
1.099  [policies/warranty.md] Todos os produtos têm 12 meses de garantia contra defeitos de fabricaç...
1.124  [policies/exchange.md] O frete da troca é gratuito, tanto na ida quanto no reenvio do novo pr...


Os números mudaram de sentido: no FAISS, o score é uma **distância** (o quadrado da distância euclidiana), então
**menor é melhor**. A ordem, porém, é a mesma. Para vetores de norma 1, as duas medidas são ligadas por uma fórmula
exata:

$$\lVert a - b \rVert^2 = 2 - 2\cos(a, b) \quad\Rightarrow\quad \cos(a, b) = 1 - \frac{d}{2}$$

In [8]:
import pandas as pd

memory_results = memory_store.similarity_search_with_score_by_vector(question_vector, k=4)
faiss_results = faiss_index.similarity_search_with_score_by_vector(question_vector, k=4)

comparison = pd.DataFrame(
    [
        {"trecho": m_doc.page_content[:45] + "...", "cosseno (InMemory)": round(m_score, 3),
         "L2² (FAISS)": round(float(f_score), 3), "1 - L2²/2": round(1 - float(f_score) / 2, 3)}
        for (m_doc, m_score), (_, f_score) in zip(memory_results, faiss_results)
    ]
)
print(comparison.to_string(index=False))

                                          trecho  cosseno (InMemory)  L2² (FAISS)  1 - L2²/2
Por arrependimento, o reembolso pode ser pedi...               0.591        0.818      0.591
Trocas podem ser solicitadas em até 30 dias c...               0.511        0.979      0.511
Todos os produtos têm 12 meses de garantia co...               0.451        1.099      0.451
O frete da troca é gratuito, tanto na ida qua...               0.438        1.124      0.438


**Observe:** a última coluna reproduz o cosseno. Com embeddings normalizados, buscar pela menor distância L2 ou pela
maior similaridade de cosseno dá **a mesma ordem**.

> ⚠️ **Armadilha comum:** copiar um limite de corte ("descarte scores abaixo de 0,5") de um *vector store* para outro.
> Cada um devolve uma medida diferente (similaridade, distância, distância ao quadrado, relevância normalizada). Antes
> de usar um limite, confira o que o score significa. No curso, preferimos decidir pela **ordem** e, no módulo 03,
> deixar o LLM avaliar a relevância.

## 4. Quantos trechos: o parâmetro `k`

`k` é quantos vizinhos a busca devolve. Vamos ver o efeito com a pergunta de frete (um vetor, reutilizado nas três
buscas):

In [9]:
shipping_question = "Qual o valor mínimo da compra para ter frete grátis?"
shipping_vector = embeddings.embed_query(shipping_question)

for k in [1, 3, 5]:
    print(f"k = {k}")
    for doc, distance in faiss_index.similarity_search_with_score_by_vector(shipping_vector, k=k):
        print(f"   {distance:.3f}  {doc.metadata['status']:<8} {doc.page_content[:75]}")

k = 1
   0.680  active   Frete grátis para compras acima de R$ 299,00 na modalidade Normal.
k = 3
   0.680  active   Frete grátis para compras acima de R$ 299,00 na modalidade Normal.
   0.710  archived Frete grátis para compras acima de R$ 199,00, em todo o Brasil.
   0.859  active   Compras acima de R$ 500,00 podem ser parceladas em até 10x sem juros no car
k = 5
   0.680  active   Frete grátis para compras acima de R$ 299,00 na modalidade Normal.
   0.710  archived Frete grátis para compras acima de R$ 199,00, em todo o Brasil.
   0.859  active   Compras acima de R$ 500,00 podem ser parceladas em até 10x sem juros no car
   1.097  active   O frete da troca é gratuito, tanto na ida quanto no reenvio do novo produto
   1.133  active   Pedidos ainda não enviados podem ser cancelados sem custo em Minha Conta.


**Observe:**

- Com `k = 1`, só a regra atual.
- Com `k = 3`, entra em segundo lugar a regra **arquivada** de R$ 199, que contradiz a primeira, e um trecho de
  parcelamento que só tem em comum "compras acima de R$".
- Com `k = 5`, chegam trechos sem relação (frete da troca, cancelamento). A busca **sempre** devolve `k` trechos,
  mesmo quando não existem `k` trechos relevantes: quem decide se servem é o passo seguinte.

| `k` pequeno | `k` grande |
|---|---|
| prompt curto e barato | mais chance de trazer o trecho certo (*recall*) |
| risco de perder a informação | mais ruído e mais tokens no prompt |

No projeto, `RETRIEVAL_K=4` (em `settings`) é o número de trechos entregues ao modelo. No módulo 02 vamos buscar mais
candidatos (`FETCH_K=12`) e reordená-los antes de escolher os 4 finais.

## 5. Filtro por metadados

A política de 2024 foi substituída, mas continua na base (é útil guardar o histórico). O que não pode acontecer é ela
chegar ao prompt. Como cada trecho carrega `status` nos metadados, a busca pode **filtrar**:

In [10]:
print("Sem filtro:")
for doc, distance in faiss_index.similarity_search_with_score_by_vector(shipping_vector, k=3):
    print(f"   {distance:.3f}  {doc.metadata['status']:<8} {doc.page_content[:75]}")

print("\nCom filter={'status': 'active'}:")
for doc, distance in faiss_index.similarity_search_with_score_by_vector(
    shipping_vector, k=3, filter={"status": "active"}
):
    print(f"   {distance:.3f}  {doc.metadata['status']:<8} {doc.page_content[:75]}")

Sem filtro:
   0.680  active   Frete grátis para compras acima de R$ 299,00 na modalidade Normal.
   0.710  archived Frete grátis para compras acima de R$ 199,00, em todo o Brasil.
   0.859  active   Compras acima de R$ 500,00 podem ser parceladas em até 10x sem juros no car

Com filter={'status': 'active'}:
   0.680  active   Frete grátis para compras acima de R$ 299,00 na modalidade Normal.
   0.859  active   Compras acima de R$ 500,00 podem ser parceladas em até 10x sem juros no car
   1.097  active   O frete da troca é gratuito, tanto na ida quanto no reenvio do novo produto


**Observe:** com o filtro, o trecho arquivado some. A regra atual continua em primeiro, mas agora sem uma regra
contraditória logo abaixo dela, pronta para confundir o modelo. As distâncias dos trechos que ficaram não mudam: o
filtro só remove candidatos.

Como o FAISS filtra:

- O filtro é um dicionário (`{"status": "active"}`, igualdade em cada chave) ou uma função que recebe os metadados e
  devolve `True`/`False`.
- O FAISS busca primeiro `fetch_k` candidatos (padrão 20), **depois** aplica o filtro e devolve até `k`. Se o filtro
  eliminar quase todos os candidatos, podem voltar menos de `k` trechos. Nesse caso, aumente `fetch_k`.

> 💡 **Dica:** na aula 02.1, `make_filter` e `CUSTOMERS_ONLY` ([src/rag/retrieval.py](../src/rag/retrieval.py))
> combinam os dois filtros que importam na TechNova: só documentos `active` e só do público `customers`.

## 6. Salvar e carregar o índice

Recalcular os embeddings a cada vez que a aplicação sobe seria desperdício. O FAISS salva o índice em uma pasta. Nesta
aula usamos uma **pasta temporária**, fora do projeto, para não tocar no índice principal (`data/faiss_index`).

In [11]:
import shutil
import tempfile
from pathlib import Path

LESSON_DIR = Path(tempfile.mkdtemp(prefix="rag_01_03_"))  # pasta temporária, apagada no fim da aula
snippets_dir = LESSON_DIR / "snippets_index"

faiss_index.save_local(str(snippets_dir))
for file in sorted(snippets_dir.iterdir()):
    print(f"{file.name:<12} {file.stat().st_size / 1024:6.1f} KB")

index.faiss   144.0 KB
index.pkl       2.7 KB


| Arquivo | Conteúdo |
|---|---|
| `index.faiss` | os vetores, no formato binário do FAISS (12 × 3072 × 4 bytes ≈ 144 KB) |
| `index.pkl` | os documentos (texto e metadados) e o mapa de ids, gravados com **pickle** |

Agora vamos carregar o índice de volta, primeiro do jeito "ingênuo":

In [12]:
try:
    FAISS.load_local(str(snippets_dir), embeddings)
except ValueError as error:
    print("ValueError:", str(error)[:230], "...")

ValueError: The de-serialization relies loading a pickle file. Pickle files can be modified to deliver a malicious payload that results in execution of arbitrary code on your machine.You will need to set `allow_dangerous_deserialization` to ` ...


O LangChain **recusa** carregar o arquivo. O motivo é o `index.pkl`: o **pickle** do Python pode executar código
arbitrário ao ser lido. Um índice baixado da internet, ou alterado por outra pessoa, poderia rodar qualquer coisa na
sua máquina. Por isso é preciso declarar, explicitamente, que você confia na origem:

In [13]:
loaded_index = FAISS.load_local(str(snippets_dir), embeddings, allow_dangerous_deserialization=True)

print("Vetores carregados:", loaded_index.index.ntotal)
doc, distance = loaded_index.similarity_search_with_score_by_vector(shipping_vector, k=1)[0]
print(f"Mesma busca, índice carregado: {distance:.3f}  {doc.page_content}")

Vetores carregados: 12
Mesma busca, índice carregado: 0.680  Frete grátis para compras acima de R$ 299,00 na modalidade Normal.


> ⚠️ **`allow_dangerous_deserialization=True` só para índices criados por você** (ou pelo seu pipeline de ingestão).
> Nunca carregue um `index.pkl` de terceiros, de um e-mail ou de um repositório público. No projeto, essa flag fica
> em um único lugar, `load_index()`, que só lê a pasta configurada em `INDEX_DIR`.

## 7. Do notebook para o projeto

### 7.1 [src/rag/vectorstore.py](../src/rag/vectorstore.py)

| Função | O que faz |
|---|---|
| `FAISS` | a classe do langchain-community, reexportada: o **único** import do community no projeto |
| `build_index(chunks, embeddings=None)` | calcula os embeddings dos trechos e cria o FAISS em memória; usa `metadata["id"]` como id quando existe (ids estáveis, aula 01.5) |
| `save_index(index, folder=None)` | `save_local` + um `index_info.json` com o modelo de embeddings, o número de trechos, a dimensão e a data |
| `load_index(folder=None, embeddings=None)` | carrega com `allow_dangerous_deserialization=True` e **avisa** se o modelo registrado for diferente do atual |
| `index_exists(folder=None)` / `index_info(folder=None)` | conferem se há índice na pasta e leem o registro, sem carregar os vetores |
| `stored_chunks(index)` | todos os trechos do índice, por id (usado na indexação incremental da aula 01.5) |

Sem `folder`, as funções usam `settings.index_dir` (`data/faiss_index`). Vamos usá-las com a pasta temporária da aula.

In [14]:
from src.rag.vectorstore import build_index, index_exists, index_info, load_index, save_index

project_dir = LESSON_DIR / "project_style_index"

index = build_index(docs)  # calcula os embeddings (uma chamada) e cria o FAISS
save_index(index, project_dir)  # grava index.faiss, index.pkl e index_info.json

print("Arquivos:", sorted(p.name for p in project_dir.iterdir()))
print("Registro:", index_info(project_dir))

reloaded = load_index(project_dir)  # confere o modelo e carrega
doc = reloaded.similarity_search_by_vector(shipping_vector, k=1, filter={"status": "active"})[0]
print("Busca no índice recarregado:", doc.page_content)

2026-09-27 21:04:56 | INFO     | src.rag.vectorstore | Criando índice FAISS | trechos=12 | embeddings=azure · text-embedding-3-large


2026-09-27 21:04:57 | INFO     | src.rag.vectorstore | Índice salvo | pasta=/private/tmp/claude-501/-Users-mitoura-Documents-LangGraph/27ed4402-274d-4728-9753-90c38a175204/scratchpad/m1/rag_01_03_ckur74an/project_style_index | trechos=12


Arquivos: ['index.faiss', 'index.pkl', 'index_info.json']
Registro: {'embeddings': 'azure · text-embedding-3-large', 'chunks': 12, 'dimensions': 3072, 'updated_at': '2026-09-27T21:04:57'}
Busca no índice recarregado: Frete grátis para compras acima de R$ 299,00 na modalidade Normal.


**Observe:** `build_index` é só um `FAISS.from_documents` com log e ids estáveis; `save_index` acrescenta o
`index_info.json`, que é a aplicação da **regra de ouro** da aula anterior: o índice carrega consigo o nome do modelo
que o criou.

### 7.2 O índice principal do projeto

O projeto já tem um índice completo em `data/faiss_index`, criado pela ingestão (`python -m src.rag.ingestion`), que
é o assunto das aulas 01.4 e 01.5. Ele tem os 34 arquivos da base divididos em trechos. Vamos só **ler** esse índice: nada aqui
o altera.

In [15]:
if not index_exists():  # o índice é criado uma vez pela ingestão (aulas 01.4 e 01.5)
    from src.rag.ingestion import index_folder

    index_folder()

main_index = load_index()
print("Registro:", index_info())
print("Trechos no índice:", main_index.index.ntotal)

q01 = "Qual o valor mínimo da compra pra ter frete grátis?"  # pergunta q01 do conjunto de avaliação
q01_vector = embeddings.embed_query(q01)

for label, filters in [("sem filtro", None), ("só active", {"status": "active"})]:
    print(f"\n{q01} ({label})")
    for doc, distance in main_index.similarity_search_with_score_by_vector(q01_vector, k=3, filter=filters):
        print(f"   {distance:.3f}  {doc.metadata['source']:<28} {doc.metadata['status']}")

Registro: {'embeddings': 'azure · text-embedding-3-large', 'chunks': 321, 'dimensions': 3072, 'updated_at': '2026-09-27T20:39:15'}
Trechos no índice: 321



Qual o valor mínimo da compra pra ter frete grátis? (sem filtro)
   0.430  faq.csv                      active
   0.653  policies/shipping_2024.md    archived
   0.703  policies/shipping_2024.md    archived

Qual o valor mínimo da compra pra ter frete grátis? (só active)
   0.430  faq.csv                      active
   0.725  policies/shipping.md         active
   0.841  faq.csv                      active


**Observe:** no índice real, a situação é pior do que no exemplo de brinquedo. Sem filtro, dois dos três primeiros
trechos vêm da política **arquivada** de 2024, e a política atual nem aparece no top 3. Com o filtro, a política atual
(`policies/shipping.md`) entra logo depois da pergunta frequente sobre frete. No exercício da próxima aula você vai ver
o efeito disso na **resposta** do LLM.

Por fim, apagamos a pasta temporária da aula. O índice principal continua intacto.

In [16]:
shutil.rmtree(LESSON_DIR)
print("Pasta temporária removida:", not LESSON_DIR.exists())
print("Índice principal intacto:", index_exists(), "|", index_info()["chunks"], "trechos")

Pasta temporária removida: True
Índice principal intacto: True | 321 trechos


## Resumo

- Um *vector store* guarda texto, vetor e metadados de cada trecho, e devolve os `k` vizinhos mais próximos de uma
  pergunta. O `InMemoryVectorStore` do langchain-core faz isso por força bruta, com similaridade de cosseno.
- O **FAISS** é a biblioteca de busca vetorial do projeto. A classe vem do `langchain-community`, que está sendo
  descontinuado; por isso o projeto a importa **só** em `src/rag/vectorstore.py` e o resto usa
  `from src.rag.vectorstore import FAISS`.
- O `IndexFlatL2` devolve a **distância L2 ao quadrado** (menor é melhor). Com vetores normalizados,
  `cos = 1 - d/2`: a ordem é a mesma do cosseno. Nunca reutilize limites de score entre *vector stores* diferentes.
- `k` equilibra *recall* e ruído; o projeto entrega 4 trechos ao modelo (`RETRIEVAL_K`).
- O **filtro por metadados** (`filter={"status": "active"}`) tira do caminho a política arquivada. O FAISS filtra
  depois de buscar `fetch_k` candidatos.
- `save_local` grava `index.faiss` e `index.pkl`; `load_local` exige `allow_dangerous_deserialization=True` porque o
  pickle pode executar código: **carregue só índices seus**.
- No produto: `build_index`, `save_index` (com `index_info.json`), `load_index` (com checagem do modelo),
  `index_exists` e `index_info`.

## Próxima aula

Até aqui, os trechos foram escritos à mão. Na aula 01.4 (`01_04_document_loading`) começamos a ingestão de verdade:
carregar os quatro formatos da base (Markdown, PDF, HTML e CSV) como `Document`, com os metadados que tornaram o
filtro desta aula possível. Depois, na aula 01.5, vêm o chunking e a indexação completa, e na 01.6 você monta o seu
primeiro RAG de ponta a ponta.